# 강의 04 · 실습 6 — 커스텀 MCP 서버 · (3.5) 디버깅

## 1. 문제상황

- 온라인 쇼핑몰 고객센터 상담원은 고객이 가격과 재고를 물으면 사내 시스템을 두 번 조회합니다.
- 가격 시스템은 판매가를 "12,000"처럼 천 단위 쉼표가 붙은 문자열로 돌려주고, 재고 시스템은 수량을 정수로 돌려줍니다.
- 상담원은 두 값을 읽고 수량을 곱해 총액을 계산한 뒤 재고가 충분한지 판단해 답합니다.
- 이 조회와 계산을 모델에게 맡기려는데, 사내 시스템의 조회 함수는 파이썬 파일 안에만 있어서 모델이 부를 수 없습니다.

## 2. 문제와 목표

- **문제**: 가격과 재고 조회 함수가 표준 규격 위에 있지 않아 모델이 부르지 못하고, 가격 시스템의 값은 쉼표가 붙은 문자열이라 그대로 계산에 쓸 수 없습니다. 아래 「6. 코드 — 스텝바이스텝」의 코드는 이 목표를 잘못 구현한 완성 코드이며, 문법 오류 없이 실행되지만 결과가 요구사항과 다른 결함 세 개를 찾아 고치는 것이 과제입니다.
- **목표**: 품목 이름을 받아 판매가와 재고를 돌려주는 도구 두 개를 MCP 도구로 등록한 서버 `shop`을 만들고, 클라이언트가 그 서버를 띄워 「무선 마우스 3개를 사면 얼마이고 재고는 충분한가」라는 질문에 두 도구를 호출해 답하게 합니다.
    - 도구 두 개: `get_price(item)`는 가격 문자열(예: `"12,000"`)에서 쉼표를 지우고 정수로 바꿔 돌려주고, `get_stock(item)`는 정수 재고 수량을 돌려줍니다.
    - 사내 데이터: 품목 이름을 가격 문자열에 대응시키는 딕셔너리 `PRICE_DB`와 재고 정수에 대응시키는 딕셔너리 `STOCK_DB`이며, 값은 아래 「6. 코드 — 스텝바이스텝」의 서버 코드 안에 있습니다.
    - 질문 한 문장은 코드에 미리 정해 넣습니다.
- **목표 달성 여부의 판정 기준**
    - 도구 목록에 `get_price`와 `get_stock`이 설명과 함께 있습니다.
    - 두 도구 호출이 모두 오류 없이 12000과 37을 돌려줍니다.
    - 최종 답이 총액 36,000원과 재고 충분을 말하는 것을 실행 결과에서 확인합니다.


## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec04_ex06_s3_diagram.svg)

## 4. 단계별 요구사항

1. **서버 인스턴스를 만듭니다.**
    - `FastMCP`로 이름이 `shop`인 서버 객체를 만듭니다.
    - 서버 파일 이름은 `shop_server.py`입니다.
2. **함수를 도구로 등록합니다.**
    - 사내 데이터는 서버 파일 안의 딕셔너리 두 개로 둡니다.
    - `PRICE_DB`는 품목 이름을 `"12,000"` 같은 문자열 판매가에, `STOCK_DB`는 품목 이름을 정수 재고 수량에 대응시킵니다.
    - `get_price(item: str) -> int`는 문자열에서 쉼표를 지우고 정수로 바꿔 돌려주고, `get_stock(item: str) -> int`는 재고 수량을 돌려줍니다.
    - 두 함수 모두 `@mcp.tool()`로 등록하고 독스트링으로 설명 한 줄을 답니다.
3. **서버를 시작합니다.**
    - `__main__` 가드 안에서 `mcp.run(transport="stdio")`를 호출합니다.
4. **클라이언트에서 띄워 확인합니다.**
    - 서버 파일을 띄우는 연결 선언으로 도구 목록을 받아 이름과 설명을 출력하고, 루프에 넣어 「무선 마우스 3개를 사면 얼마이고 재고는 충분한가요?」를 물어 도구 호출·도구 결과 상태·최종 답을 출력합니다.
    - 도구 결과 메시지의 `status`가 `'success'`이면 성공, `'error'`이면 실패입니다.
    - 도구 목록은 「서버가 준 도구:」 줄로 출력합니다.


## 5. 코드 골격 — MCP 서버 3단

FastMCP로 서버를 세우는 순서는 다음 세 단계입니다. 아래 「6. 코드 — 스텝바이스텝」의 코드 셀이 이 세 단계와 하나씩 대응합니다. 서버 코드는 `%%writefile`로 파일에 쓰고, 단계 ③의 확인 셀에서 클라이언트가 그 파일을 실행 명령으로 띄웁니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 인스턴스 생성 | 서버 객체를 이름과 함께 만듭니다 | `mcp = FastMCP("shop")` | 1 |
| ② 도구 등록 | 파이썬 함수에 표시를 붙이고, 타입 힌트와 설명 한 줄을 답니다 | `@mcp.tool()`, `def get_price(item: str) -> int` | 2 |
| ③ 서버 시작 | 표준입출력으로 말하도록 지정해 서버를 띄웁니다. 클라이언트가 실행 명령으로 띄우고 도구 목록을 받아 씁니다 | `mcp.run(transport="stdio")`, `MultiServerMCPClient`, `get_tools()` | 3, 4 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

클라이언트 쪽 준비입니다. 라이브러리를 불러오고 모델을 준비하고, 도구 호출 루프 `build_loop`와 연결 선언 함수 `server_config`를 정의합니다.

- API 키는 `.env` 파일에서 읽습니다. `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 두고, 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

- `build_loop`는 받아 온 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결하는 도구 호출 루프입니다. 서버 코드가 아니라 서버를 쓰는 쪽의 코드입니다.
- `sys.stderr = sys.__stderr__` 줄은 노트북 전용입니다. 노트북 커널은 표준 오류 스트림을 화면용 객체로 바꿔 두는데, 서버 프로세스를 띄우는 코드는 원래의 표준 오류 스트림을 요구하므로 되돌려 놓습니다. 이 줄은 MCP 클라이언트를 불러오기 전에 있어야 합니다.
- `server_config`는 서버 파일 하나를 표준입출력으로 띄우는 연결 선언입니다. `sys.executable`은 지금 돌고 있는 파이썬 러너입니다. `FASTMCP_LOG_LEVEL`은 서버의 안내 로그가 화면을 채우지 않게 하는 설정입니다.

In [ ]:
import os
import sys
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from typing import Annotated, TypedDict

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

sys.stderr = sys.__stderr__   # 노트북 커널의 stderr에는 fileno()가 없어 서버 프로세스 시작이 실패하므로 원래 stderr로 되돌린다
from langchain_mcp_adapters.client import MultiServerMCPClient
from langgraph.graph import END, START, StateGraph
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")

llm = init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")


class State(TypedDict):
    messages: Annotated[list, add_messages]


def build_loop(tools):
    """도구 호출 루프. 도구를 모델에 묶고 model·tools 두 노드를 조건부 엣지로 연결한다."""
    bound = llm.bind_tools(tools)

    def call_model(state: State) -> dict:
        return {"messages": [bound.invoke(state["messages"])]}

    def should_continue(state: State) -> str:
        return "tools" if state["messages"][-1].tool_calls else END

    g = StateGraph(State)
    g.add_node("model", call_model)
    g.add_node("tools", ToolNode(tools))
    g.add_edge(START, "model")
    g.add_conditional_edges("model", should_continue, {"tools": "tools", END: END})
    g.add_edge("tools", "model")
    return g.compile()


def text_of(m) -> str:
    """메시지 내용이 콘텐츠 블록 목록이면 글자 부분만 이어 붙인다."""
    if isinstance(m.content, list):
        return " ".join(p.get("text", "") for p in m.content if isinstance(p, dict))
    return str(m.content)


def show(result) -> None:
    """실행 결과의 메시지를 종류·도구 호출·상태와 함께 한 줄씩 출력한다."""
    for m in result["messages"]:
        kind = type(m).__name__
        calls = getattr(m, "tool_calls", None)
        if calls:
            print(f"[{kind}] tool_calls={[(c['name'], c['args']) for c in calls]}")
        elif kind == "ToolMessage":
            print(f"[{kind}] status={m.status!r} {text_of(m)[:160]}")
        elif m.content:
            print(f"[{kind}] {text_of(m)[:300]}")


def server_config(file: str) -> dict:
    """서버 파일 하나를 표준입출력으로 띄우는 연결 선언을 만든다."""
    return {"command": sys.executable, "args": [str(Path(file).resolve())],
            "transport": "stdio", "env": {"FASTMCP_LOG_LEVEL": "ERROR"}}


print("클라이언트 준비를 마쳤습니다.")

### 단계 ① — 인스턴스 생성 (요구사항 1)

`FastMCP(이름)`이 서버 한 대입니다. 괄호 안 이름이 이 서버의 이름입니다. `%%writefile`이 이 셀의 내용을 서버 파일로 저장합니다. 서버 파일은 사람이 직접 실행하지 않고, 단계 ③에서 클라이언트가 띄웁니다.

In [ ]:
%%writefile shop_server_bug.py
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("shop")

### 단계 ② — 도구 등록 (요구사항 2)

`@mcp.tool()` 한 줄을 얹으면 그 파이썬 함수가 MCP 도구가 됩니다. 함수 이름이 도구 이름, 독스트링이 도구 설명, 인자와 반환의 타입 힌트가 도구 규격으로 그대로 나갑니다. `%%writefile -a`는 서버 파일 뒤에 이어 붙입니다.

In [ ]:
%%writefile -a shop_server_bug.py

PRICE_DB = {"무선 마우스": "12,000", "키보드": "45,000", "모니터": "210,000"}   # 가격 시스템의 원래 값
STOCK_DB = {"무선 마우스": 37, "키보드": 0, "모니터": 5}                          # 재고 시스템의 값


@mcp.tool()
def get_price(item: str) -> int:
    return PRICE_DB[item]


def get_stock(item: str) -> int:
    """품목의 현재 재고 수량을 돌려준다."""
    return STOCK_DB[item]

### 단계 ③ — 서버 시작 (요구사항 3, 4)

`transport="stdio"`는 표준입출력으로 말한다는 뜻입니다. 서버는 포트를 열지 않고, 자신을 실행한 쪽과 입출력으로 주고받습니다. 아래 두 셀(③-a, ③-b)이 모두 이 단계에 속합니다.

#### 단계 ③-a — 시작 코드를 서버 파일에 붙입니다

In [ ]:
%%writefile -a shop_server_bug.py

if __name__ == "__main__":
    mcp.run(transport="stdio")

#### 단계 ③-b — 클라이언트에서 띄우고 확인합니다

클라이언트에 적는 것은 서버를 띄우는 실행 명령뿐입니다. 서버 코드는 클라이언트에 등장하지 않습니다. `get_tools()`가 서버가 내놓은 도구 목록을 받아 오고, 그 목록을 `build_loop`에 그대로 넣습니다.

In [ ]:
print(Path("shop_server_bug.py").read_text(encoding="utf-8"))

client = MultiServerMCPClient({"shop": server_config("shop_server_bug.py")})
tools = await client.get_tools()
print("서버가 준 도구:", [(t.name, t.description) for t in tools])

app = build_loop(tools)
out = await app.ainvoke({"messages": [HumanMessage("무선 마우스 3개를 사면 얼마이고 재고는 충분한가요? 도구로 확인해줘.")]})
show(out)

## 7. 실행 결과 확인

결함 코드를 실행했을 때의 증상과, 고친 뒤 통과해야 하는 항목입니다.

1. **증상 1**: `서버가 준 도구:` 줄에 도구가 하나뿐입니다. 요구사항 2는 도구 두 개를 등록하라고 했습니다.
2. **증상 2**: 그 하나뿐인 도구의 설명이 빈 문자열입니다. 요구사항 2는 독스트링으로 설명을 달라고 했습니다.
3. **증상 3**: `get_price`의 `ToolMessage`가 `status='error'`이고 내용에 `int_parsing`과 `input_value='12,000'`이 보입니다. 선언은 `int`인데 실제 반환값이 문자열이라 서버 쪽 검증에서 막혔습니다. 그런데도 마지막 `AIMessage`는 오류 문자열 안의 숫자를 읽어 그럴듯한 총액을 답합니다. 최종 답이 맞아 보여도 도구는 실패했습니다. 재고는 도구가 없어 확인하지 못했다고 하거나 지어낸 값을 말합니다.
4. **고친 뒤**: 도구 목록에 두 도구가 설명과 함께 있고, 두 `ToolMessage`가 모두 `status='success'`이며 값이 12000과 37이고, 최종 답이 36,000원과 재고 충분을 말합니다.